In [100]:
import numpy as np
import scipy.interpolate as terp
k = 1.3806503e-23
J_mev = 1.6021766e-13
cm_fm = 1e-13
erg_J = 1e7
c = 2.99792458e10
kscale = 1e-15/c**2

In [101]:
data = np.loadtxt("eos.table")

nb_fm3 = data[:,1]
P_mev_fm3 = data[:,3]
rho_mev_fm3 = data[:,4]

In [102]:
nb_cgs = nb_fm3 / cm_fm**3
P_cgs = P_mev_fm3 * J_mev*erg_J/cm_fm**3
rho_cgs = rho_mev_fm3 * J_mev*erg_J/c**2/cm_fm**3

In [92]:
def integrate_h(rho, P):

    indices = np.where(P > 0)[0]

    rho = (rho[indices]*c**2*kscale)
    P = (P[indices]*kscale)

    N = 16000
    h = np.zeros(rho.size)
    cs_rho = terp.CubicSpline(np.arange(rho.size), rho)
    cs_P = terp.CubicSpline(np.arange(P.size), P)

    for i in range(rho.size):
        dP = P[i]/N
        sum = 0
        rho_terp = cs_rho(np.linspace(0, i, N))
        P_terp = cs_P(np.linspace(0, i, N))
        for k in range(N):
            dH = dP/(1+rho_terp[k]+P_terp[k])
            sum += dH
        h[i] = sum*np.log(10)

    return h

In [93]:
h_cgs = integrate_h(rho_cgs, P_cgs)

In [94]:
h_cgs

array([2.30258509e+00, 2.42278648e+00, 2.55287456e+00, 2.69373631e+00,
       2.84635821e+00, 3.01181313e+00, 3.19127078e+00, 3.38600535e+00,
       3.59740360e+00, 3.82697334e+00, 4.07635209e+00, 4.34731586e+00,
       4.64178813e+00, 4.96184858e+00, 5.30974176e+00, 5.68788524e+00,
       6.09887713e+00, 6.54550268e+00, 7.03073976e+00, 7.55776270e+00,
       8.12994441e+00, 8.75085623e+00, 9.42426533e+00, 1.01541295e+01,
       1.09445887e+01, 1.17999549e+01, 1.27246979e+01, 1.37234316e+01,
       1.48008990e+01, 1.59619603e+01, 1.72115871e+01, 1.85548690e+01,
       1.99970390e+01, 2.15435300e+01, 2.32000770e+01, 2.49728853e+01,
       2.68712897e+01, 2.89153192e+01, 3.11324729e+01, 3.35572338e+01,
       3.62317350e+01, 3.92059360e+01, 4.25369882e+01, 4.62873854e+01,
       5.05214348e+01, 5.52996000e+01, 6.06704369e+01, 6.66603310e+01,
       7.32623044e+01, 8.04272394e+01, 8.80646680e+01, 9.60669789e+01,
       1.04382342e+02, 1.13160427e+02, 1.22620564e+02, 1.32884806e+02,
      

In [105]:
newtable = np.zeros([500,2])
newtable[:,0] = rho_cgs
newtable[:,1] = P_cgs

In [106]:
np.savetxt("eos2.table", newtable)